<a href="https://colab.research.google.com/github/Henix285/telecom-churnprediction/blob/main/notebooks/03_temporal_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# TELECOM CHURN PREDICTION
# TEMPORAL MODELING
# ============================================================

print("=" * 60)
print("TELECOM CHURN PREDICTION - TEMPORAL MODELING")
print("=" * 60)
print("Models: LSTM and GRU")
print("Objective: Predict customer inactivity from monthly behavior")

TELECOM CHURN PREDICTION - TEMPORAL MODELING
Models: LSTM and GRU
Objective: Predict customer inactivity from monthly behavior


In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.20.0
GPU: []


In [3]:
print("Current directory:")
print(os.getcwd())

Current directory:
/content


In [4]:
!git clone https://github.com/Henix285/telecom-churnprediction.git

Cloning into 'telecom-churnprediction'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 77 (delta 28), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 1.26 MiB | 3.27 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [5]:
%cd /content/telecom-churnprediction

/content/telecom-churnprediction


In [6]:
!find . -maxdepth 3 -type f | sort

./03_temporal_models.ipynb
./figures/shap_customer_7000040799.png
./figures/shap_global_importance.png
./figures/shap_summary.png
./.git/config
./.git/description
./.git/HEAD
./.git/hooks/applypatch-msg.sample
./.git/hooks/commit-msg.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/post-update.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/pre-commit.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/prepare-commit-msg.sample
./.git/hooks/pre-push.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-receive.sample
./.git/hooks/push-to-checkout.sample
./.git/hooks/sendemail-validate.sample
./.git/hooks/update.sample
./.gitignore
./.git/index
./.git/info/exclude
./.git/logs/HEAD
./.git/packed-refs
./notebooks/01_data_preprocessing.ipynb
./notebooks/03_temporal_models.ipynb
./README.md
./requirements.txt
./results/customer_risk_predictions.csv
./results/model_features.csv
./results/model_metrics.json
./results/risk_group_summary.csv
./results/shap_feature_importa

In [7]:
print("Files in notebooks folder:")
print("=" * 50)

for file in os.listdir("notebooks"):
    print(file)

Files in notebooks folder:
01_data_preprocessing.ipynb
03_temporal_models.ipynb


In [8]:
print("Searching for telecom_churn_data.csv...")

matches = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "telecom_churn_data.csv":
            matches.append(os.path.join(root, file))

if matches:
    print("\nFound:")
    for path in matches:
        print(path)
else:
    print("\ntelecom_churn_data.csv NOT FOUND")

Searching for telecom_churn_data.csv...

telecom_churn_data.csv NOT FOUND


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:
df = pd.read_csv(
    "/content/telecom-churnprediction/telecom_churn_data.csv"
)

In [14]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 99999
Columns: 226


In [15]:
df.head()

,mobile_number,circle_id,loc_og_t2o_mou,std_og_t2o_mou,loc_ic_t2o_mou,last_date_of_month_6,last_date_of_month_7,last_date_of_month_8,last_date_of_month_9,arpu_6,...,sachet_3g_9,fb_user_6,fb_user_7,fb_user_8,fb_user_9,aon,aug_vbc_3g,jul_vbc_3g,jun_vbc_3g,sep_vbc_3g
0,7000842753,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,9/30/2014,197.385,...,0,1.0,1.0,1.0,NaN,968,30.4,0.0,101.20,3.58
1,7001865778,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,9/30/2014,34.047,...,0,NaN,1.0,1.0,NaN,1006,0.0,0.0,0.00,0.00
2,7001625959,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,9/30/2014,167.690,...,0,NaN,NaN,NaN,1.0,1103,0.0,0.0,4.17,0.00
3,7001204172,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,9/30/2014,221.338,...,0,NaN,NaN,NaN,NaN,2491,0.0,0.0,0.00,0.00
4,7000142493,109,0.0,0.0,0.0,6/30/2014,7/31/2014,8/31/2014,9/30/2014,261.636,...,0,0.0,NaN,NaN,NaN,1526,0.0,0.0,0.00,0.00


In [16]:
print(df.columns.tolist())

['mobile_number', 'circle_id', 'loc_og_t2o_mou', 'std_og_t2o_mou', 'loc_ic_t2o_mou', 'last_date_of_month_6', 'last_date_of_month_7', 'last_date_of_month_8', 'last_date_of_month_9', 'arpu_6', 'arpu_7', 'arpu_8', 'arpu_9', 'onnet_mou_6', 'onnet_mou_7', 'onnet_mou_8', 'onnet_mou_9', 'offnet_mou_6', 'offnet_mou_7', 'offnet_mou_8', 'offnet_mou_9', 'roam_ic_mou_6', 'roam_ic_mou_7', 'roam_ic_mou_8', 'roam_ic_mou_9', 'roam_og_mou_6', 'roam_og_mou_7', 'roam_og_mou_8', 'roam_og_mou_9', 'loc_og_t2t_mou_6', 'loc_og_t2t_mou_7', 'loc_og_t2t_mou_8', 'loc_og_t2t_mou_9', 'loc_og_t2m_mou_6', 'loc_og_t2m_mou_7', 'loc_og_t2m_mou_8', 'loc_og_t2m_mou_9', 'loc_og_t2f_mou_6', 'loc_og_t2f_mou_7', 'loc_og_t2f_mou_8', 'loc_og_t2f_mou_9', 'loc_og_t2c_mou_6', 'loc_og_t2c_mou_7', 'loc_og_t2c_mou_8', 'loc_og_t2c_mou_9', 'loc_og_mou_6', 'loc_og_mou_7', 'loc_og_mou_8', 'loc_og_mou_9', 'std_og_t2t_mou_6', 'std_og_t2t_mou_7', 'std_og_t2t_mou_8', 'std_og_t2t_mou_9', 'std_og_t2m_mou_6', 'std_og_t2m_mou_7', 'std_og_t2m_mou

In [17]:
month_6 = [c for c in df.columns if c.endswith("_6")]
month_7 = [c for c in df.columns if c.endswith("_7")]
month_8 = [c for c in df.columns if c.endswith("_8")]

print("June (_6) columns:", len(month_6))
print("July (_7) columns:", len(month_7))
print("August (_8) columns:", len(month_8))

June (_6) columns: 54
July (_7) columns: 54
August (_8) columns: 54


In [18]:
print("June examples:")
print(month_6[:20])

print("\nJuly examples:")
print(month_7[:20])

print("\nAugust examples:")
print(month_8[:20])

June examples:
['last_date_of_month_6', 'arpu_6', 'onnet_mou_6', 'offnet_mou_6', 'roam_ic_mou_6', 'roam_og_mou_6', 'loc_og_t2t_mou_6', 'loc_og_t2m_mou_6', 'loc_og_t2f_mou_6', 'loc_og_t2c_mou_6', 'loc_og_mou_6', 'std_og_t2t_mou_6', 'std_og_t2m_mou_6', 'std_og_t2f_mou_6', 'std_og_t2c_mou_6', 'std_og_mou_6', 'isd_og_mou_6', 'spl_og_mou_6', 'og_others_6', 'total_og_mou_6']

July examples:
['last_date_of_month_7', 'arpu_7', 'onnet_mou_7', 'offnet_mou_7', 'roam_ic_mou_7', 'roam_og_mou_7', 'loc_og_t2t_mou_7', 'loc_og_t2m_mou_7', 'loc_og_t2f_mou_7', 'loc_og_t2c_mou_7', 'loc_og_mou_7', 'std_og_t2t_mou_7', 'std_og_t2m_mou_7', 'std_og_t2f_mou_7', 'std_og_t2c_mou_7', 'std_og_mou_7', 'isd_og_mou_7', 'spl_og_mou_7', 'og_others_7', 'total_og_mou_7']

August examples:
['last_date_of_month_8', 'arpu_8', 'onnet_mou_8', 'offnet_mou_8', 'roam_ic_mou_8', 'roam_og_mou_8', 'loc_og_t2t_mou_8', 'loc_og_t2m_mou_8', 'loc_og_t2f_mou_8', 'loc_og_t2c_mou_8', 'loc_og_mou_8', 'std_og_t2t_mou_8', 'std_og_t2m_mou_8', '

In [19]:
target_candidates = [
    c for c in df.columns
    if any(word in c.lower() for word in ["churn", "inactive", "target"])
]

print("Possible target columns:")
print(target_candidates)

Possible target columns:
[]


In [22]:
import json

notebook_path = "/content/telecom-churnprediction/notebooks/01_data_preprocessing.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

target_lines = []

for cell in nb["cells"]:
    source = "".join(cell.get("source", []))

    if any(word in source.lower() for word in [
        "target",
        "inactive",
        "churn",
        "newly_inactive"
    ]):
        target_lines.append(source)

print("\n\n".join(target_lines))

df = pd.read_csv("/content/telecom_churn_data.csv")

print("Dataset shape:", df.shape)
df.head()

# Search for columns that may represent churn / target

possible_target_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in
           ['churn', 'churned', 'target', 'label', 'retention'])
]

print("Possible target columns:")
print(possible_target_cols)

# ==========================================
# STEP 4: Identify completely inactive users
# ==========================================

# Treat missing activity values as zero
og_sep = df['total_og_mou_9'].fillna(0)
ic_sep = df['total_ic_mou_9'].fillna(0)
data_2g_sep = df['vol_2g_mb_9'].fillna(0)
data_3g_sep = df['vol_3g_mb_9'].fillna(0)

# Voice inactivity
voice_inactive = (og_sep == 0) & (ic_sep == 0)

# Complete inactivity: no outgoing/incoming voice AND no 2G/3G usage
complete_inactive = (
    (og_sep == 0) &
    (ic_sep == 0) &
    (data_2g_sep == 0) &
    (data_3g_sep == 0)
)

print("Voice-inactive custo

In [23]:
import os

print(os.path.exists(
    "/content/telecom-churnprediction/notebooks/01_data_preprocessing.ipynb"
))

True


In [24]:
sep_columns = [c for c in df.columns if c.endswith("_9")]

print("September (_9) columns:", len(sep_columns))
print(sep_columns[:50])

September (_9) columns: 54
['last_date_of_month_9', 'arpu_9', 'onnet_mou_9', 'offnet_mou_9', 'roam_ic_mou_9', 'roam_og_mou_9', 'loc_og_t2t_mou_9', 'loc_og_t2m_mou_9', 'loc_og_t2f_mou_9', 'loc_og_t2c_mou_9', 'loc_og_mou_9', 'std_og_t2t_mou_9', 'std_og_t2m_mou_9', 'std_og_t2f_mou_9', 'std_og_t2c_mou_9', 'std_og_mou_9', 'isd_og_mou_9', 'spl_og_mou_9', 'og_others_9', 'total_og_mou_9', 'loc_ic_t2t_mou_9', 'loc_ic_t2m_mou_9', 'loc_ic_t2f_mou_9', 'loc_ic_mou_9', 'std_ic_t2t_mou_9', 'std_ic_t2m_mou_9', 'std_ic_t2f_mou_9', 'std_ic_t2o_mou_9', 'std_ic_mou_9', 'total_ic_mou_9', 'spl_ic_mou_9', 'isd_ic_mou_9', 'ic_others_9', 'total_rech_num_9', 'total_rech_amt_9', 'max_rech_amt_9', 'date_of_last_rech_9', 'last_day_rch_amt_9', 'date_of_last_rech_data_9', 'total_rech_data_9', 'max_rech_data_9', 'count_rech_2g_9', 'count_rech_3g_9', 'av_rech_amt_data_9', 'vol_2g_mb_9', 'vol_3g_mb_9', 'arpu_3g_9', 'arpu_2g_9', 'night_pck_user_9', 'monthly_2g_9']


In [25]:
date_columns = [
    c for c in df.columns
    if "date" in c.lower()
]

print("Date columns:")
print(date_columns)

Date columns:
['last_date_of_month_6', 'last_date_of_month_7', 'last_date_of_month_8', 'last_date_of_month_9', 'date_of_last_rech_6', 'date_of_last_rech_7', 'date_of_last_rech_8', 'date_of_last_rech_9', 'date_of_last_rech_data_6', 'date_of_last_rech_data_7', 'date_of_last_rech_data_8', 'date_of_last_rech_data_9']


In [26]:
for col in date_columns:
    print("\n", col)
    print(df[col].dropna().astype(str).value_counts().head(10))


 last_date_of_month_6
last_date_of_month_6
6/30/2014    99999
Name: count, dtype: int64

 last_date_of_month_7
last_date_of_month_7
7/31/2014    99398
Name: count, dtype: int64

 last_date_of_month_8
last_date_of_month_8
8/31/2014    98899
Name: count, dtype: int64

 last_date_of_month_9
last_date_of_month_9
9/30/2014    98340
Name: count, dtype: int64

 date_of_last_rech_6
date_of_last_rech_6
6/30/2014    16960
6/29/2014    12918
6/27/2014    11169
6/28/2014     9491
6/26/2014     5530
6/25/2014     4896
6/17/2014     4145
6/24/2014     4129
6/14/2014     3845
6/21/2014     3747
Name: count, dtype: int64

 date_of_last_rech_7
date_of_last_rech_7
7/31/2014    17288
7/30/2014    13863
7/25/2014     9401
7/29/2014     9052
7/28/2014     7502
7/27/2014     5909
7/26/2014     5382
7/24/2014     3998
7/19/2014     3057
7/22/2014     2969
Name: count, dtype: int64

 date_of_last_rech_8
date_of_last_rech_8
8/31/2014    14706
8/30/2014    11707
8/29/2014    10057
8/28/2014     9816
8/26/2014 

In [27]:
features_6 = {
    c.rsplit("_", 1)[0]
    for c in month_6
    if c != "last_date_of_month_6"
}

features_7 = {
    c.rsplit("_", 1)[0]
    for c in month_7
    if c != "last_date_of_month_7"
}

features_8 = {
    c.rsplit("_", 1)[0]
    for c in month_8
    if c != "last_date_of_month_8"
}

common_features = sorted(
    features_6 & features_7 & features_8
)

print("Common monthly features:", len(common_features))
print(common_features)

Common monthly features: 53
['arpu', 'arpu_2g', 'arpu_3g', 'av_rech_amt_data', 'count_rech_2g', 'count_rech_3g', 'date_of_last_rech', 'date_of_last_rech_data', 'fb_user', 'ic_others', 'isd_ic_mou', 'isd_og_mou', 'last_day_rch_amt', 'loc_ic_mou', 'loc_ic_t2f_mou', 'loc_ic_t2m_mou', 'loc_ic_t2t_mou', 'loc_og_mou', 'loc_og_t2c_mou', 'loc_og_t2f_mou', 'loc_og_t2m_mou', 'loc_og_t2t_mou', 'max_rech_amt', 'max_rech_data', 'monthly_2g', 'monthly_3g', 'night_pck_user', 'offnet_mou', 'og_others', 'onnet_mou', 'roam_ic_mou', 'roam_og_mou', 'sachet_2g', 'sachet_3g', 'spl_ic_mou', 'spl_og_mou', 'std_ic_mou', 'std_ic_t2f_mou', 'std_ic_t2m_mou', 'std_ic_t2o_mou', 'std_ic_t2t_mou', 'std_og_mou', 'std_og_t2c_mou', 'std_og_t2f_mou', 'std_og_t2m_mou', 'std_og_t2t_mou', 'total_ic_mou', 'total_og_mou', 'total_rech_amt', 'total_rech_data', 'total_rech_num', 'vol_2g_mb', 'vol_3g_mb']


In [28]:
monthly_columns = (
    [c for c in month_6 if c != "last_date_of_month_6"] +
    [c for c in month_7 if c != "last_date_of_month_7"] +
    [c for c in month_8 if c != "last_date_of_month_8"]
)

missing_summary = df[monthly_columns].isna().mean().sort_values(
    ascending=False
)

print("Top 20 columns by missing percentage:")
print((missing_summary.head(20) * 100).round(2))

Top 20 columns by missing percentage:
count_rech_2g_6             74.85
count_rech_3g_6             74.85
av_rech_amt_data_6          74.85
arpu_2g_6                   74.85
arpu_3g_6                   74.85
night_pck_user_6            74.85
fb_user_6                   74.85
max_rech_data_6             74.85
date_of_last_rech_data_6    74.85
total_rech_data_6           74.85
count_rech_3g_7             74.43
total_rech_data_7           74.43
count_rech_2g_7             74.43
max_rech_data_7             74.43
date_of_last_rech_data_7    74.43
av_rech_amt_data_7          74.43
arpu_3g_7                   74.43
night_pck_user_7            74.43
arpu_2g_7                   74.43
fb_user_7                   74.43
dtype: float64


In [29]:
print("First 20 columns:")
print(df.columns[:20].tolist())

First 20 columns:
['mobile_number', 'circle_id', 'loc_og_t2o_mou', 'std_og_t2o_mou', 'loc_ic_t2o_mou', 'last_date_of_month_6', 'last_date_of_month_7', 'last_date_of_month_8', 'last_date_of_month_9', 'arpu_6', 'arpu_7', 'arpu_8', 'arpu_9', 'onnet_mou_6', 'onnet_mou_7', 'onnet_mou_8', 'onnet_mou_9', 'offnet_mou_6', 'offnet_mou_7', 'offnet_mou_8']


In [30]:
print(df.dtypes.value_counts())

float64    179
int64       35
object      12
Name: count, dtype: int64


In [31]:
PROJECT = "/content/telecom-churnprediction"
print(PROJECT)

/content/telecom-churnprediction


In [32]:
import json

notebook_path = f"{PROJECT}/notebooks/01_data_preprocessing.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    if any(word in source.lower() for word in [
        "newly_inactive",
        "inactive",
        "target"
    ]):
        print(f"\n{'='*70}")
        print(f"CELL {i}")
        print("="*70)
        print(source)


CELL 13
# Search for columns that may represent churn / target

possible_target_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in
           ['churn', 'churned', 'target', 'label', 'retention'])
]

print("Possible target columns:")
print(possible_target_cols)

CELL 17
# ==========================================
# STEP 4: Identify completely inactive users
# ==========================================

# Treat missing activity values as zero
og_sep = df['total_og_mou_9'].fillna(0)
ic_sep = df['total_ic_mou_9'].fillna(0)
data_2g_sep = df['vol_2g_mb_9'].fillna(0)
data_3g_sep = df['vol_3g_mb_9'].fillna(0)

# Voice inactivity
voice_inactive = (og_sep == 0) & (ic_sep == 0)

# Complete inactivity: no outgoing/incoming voice AND no 2G/3G usage
complete_inactive = (
    (og_sep == 0) &
    (ic_sep == 0) &
    (data_2g_sep == 0) &
    (data_3g_sep == 0)
)

print("Voice-inactive customers:", voice_inactive.sum())
print("Completely inactive customers:", complete_in

In [33]:
sep_columns = [c for c in df.columns if c.endswith("_9")]

print("Number of September columns:", len(sep_columns))
print("\nSeptember columns:")
print(sep_columns)

Number of September columns: 54

September columns:
['last_date_of_month_9', 'arpu_9', 'onnet_mou_9', 'offnet_mou_9', 'roam_ic_mou_9', 'roam_og_mou_9', 'loc_og_t2t_mou_9', 'loc_og_t2m_mou_9', 'loc_og_t2f_mou_9', 'loc_og_t2c_mou_9', 'loc_og_mou_9', 'std_og_t2t_mou_9', 'std_og_t2m_mou_9', 'std_og_t2f_mou_9', 'std_og_t2c_mou_9', 'std_og_mou_9', 'isd_og_mou_9', 'spl_og_mou_9', 'og_others_9', 'total_og_mou_9', 'loc_ic_t2t_mou_9', 'loc_ic_t2m_mou_9', 'loc_ic_t2f_mou_9', 'loc_ic_mou_9', 'std_ic_t2t_mou_9', 'std_ic_t2m_mou_9', 'std_ic_t2f_mou_9', 'std_ic_t2o_mou_9', 'std_ic_mou_9', 'total_ic_mou_9', 'spl_ic_mou_9', 'isd_ic_mou_9', 'ic_others_9', 'total_rech_num_9', 'total_rech_amt_9', 'max_rech_amt_9', 'date_of_last_rech_9', 'last_day_rch_amt_9', 'date_of_last_rech_data_9', 'total_rech_data_9', 'max_rech_data_9', 'count_rech_2g_9', 'count_rech_3g_9', 'av_rech_amt_data_9', 'vol_2g_mb_9', 'vol_3g_mb_9', 'arpu_3g_9', 'arpu_2g_9', 'night_pck_user_9', 'monthly_2g_9', 'sachet_2g_9', 'monthly_3g_9', 

In [34]:
possible_id_columns = [
    c for c in df.columns
    if any(word in c.lower() for word in [
        "mobile", "customer", "id"
    ])
]

print("Possible customer ID columns:")
print(possible_id_columns)

Possible customer ID columns:
['mobile_number', 'circle_id']


In [35]:
month6_features = [
    c[:-2] for c in df.columns
    if c.endswith("_6") and c != "last_date_of_month_6"
]

month7_features = [
    c[:-2] for c in df.columns
    if c.endswith("_7") and c != "last_date_of_month_7"
]

month8_features = [
    c[:-2] for c in df.columns
    if c.endswith("_8") and c != "last_date_of_month_8"
]

common_features = sorted(
    set(month6_features)
    & set(month7_features)
    & set(month8_features)
)

print("Common features:", len(common_features))
print(common_features)

Common features: 53
['arpu', 'arpu_2g', 'arpu_3g', 'av_rech_amt_data', 'count_rech_2g', 'count_rech_3g', 'date_of_last_rech', 'date_of_last_rech_data', 'fb_user', 'ic_others', 'isd_ic_mou', 'isd_og_mou', 'last_day_rch_amt', 'loc_ic_mou', 'loc_ic_t2f_mou', 'loc_ic_t2m_mou', 'loc_ic_t2t_mou', 'loc_og_mou', 'loc_og_t2c_mou', 'loc_og_t2f_mou', 'loc_og_t2m_mou', 'loc_og_t2t_mou', 'max_rech_amt', 'max_rech_data', 'monthly_2g', 'monthly_3g', 'night_pck_user', 'offnet_mou', 'og_others', 'onnet_mou', 'roam_ic_mou', 'roam_og_mou', 'sachet_2g', 'sachet_3g', 'spl_ic_mou', 'spl_og_mou', 'std_ic_mou', 'std_ic_t2f_mou', 'std_ic_t2m_mou', 'std_ic_t2o_mou', 'std_ic_t2t_mou', 'std_og_mou', 'std_og_t2c_mou', 'std_og_t2f_mou', 'std_og_t2m_mou', 'std_og_t2t_mou', 'total_ic_mou', 'total_og_mou', 'total_rech_amt', 'total_rech_data', 'total_rech_num', 'vol_2g_mb', 'vol_3g_mb']


In [36]:
print("June features :", len(month6_features))
print("July features :", len(month7_features))
print("August features:", len(month8_features))
print("Common features:", len(common_features))

print("\nSame feature set across all 3 months:",
      set(month6_features) == set(month7_features) == set(month8_features))

June features : 53
July features : 53
August features: 53
Common features: 53

Same feature set across all 3 months: True


In [37]:
sample_features = common_features[:5]

print("Sample temporal data:\n")

for feature in sample_features:
    print(f"\n{feature}")
    print("June  :", df[f"{feature}_6"].head(3).tolist())
    print("July  :", df[f"{feature}_7"].head(3).tolist())
    print("August:", df[f"{feature}_8"].head(3).tolist())

Sample temporal data:


arpu
June  : [197.385, 34.047, 167.69]
July  : [214.816, 355.074, 189.058]
August: [213.803, 268.321, 210.226]

arpu_2g
June  : [212.17, nan, nan]
July  : [212.17, 28.61, nan]
August: [212.17, 7.6, nan]

arpu_3g
June  : [212.17, nan, nan]
July  : [212.17, 0.0, nan]
August: [212.17, 0.0, nan]

av_rech_amt_data
June  : [252.0, nan, nan]
July  : [252.0, 154.0, nan]
August: [252.0, 50.0, nan]

count_rech_2g
June  : [0.0, nan, nan]
July  : [0.0, 1.0, nan]
August: [0.0, 2.0, nan]


In [38]:
temporal_columns = []

for feature in common_features:
    temporal_columns.extend([
        f"{feature}_6",
        f"{feature}_7",
        f"{feature}_8"
    ])

missing = df[temporal_columns].isna().sum()

print("Total missing values:", missing.sum())

print("\nTop 15 columns with missing values:")
print(missing.sort_values(ascending=False).head(15))

Total missing values: 2618382

Top 15 columns with missing values:
arpu_2g_6                   74846
arpu_3g_6                   74846
av_rech_amt_data_6          74846
count_rech_2g_6             74846
fb_user_6                   74846
date_of_last_rech_data_6    74846
count_rech_3g_6             74846
max_rech_data_6             74846
total_rech_data_6           74846
night_pck_user_6            74846
count_rech_2g_7             74428
date_of_last_rech_data_7    74428
av_rech_amt_data_7          74428
count_rech_3g_7             74428
fb_user_7                   74428
dtype: int64


In [42]:
# Check the actual columns related to voice activity

voice_columns = [
    col for col in df.columns
    if 'voice' in col.lower()
]

print("Voice-related columns:")
print(voice_columns)

Voice-related columns:
[]


In [43]:
print("\nColumns ending in _8:")
print([col for col in df.columns if col.endswith('_8')])

print("\nColumns ending in _9:")
print([col for col in df.columns if col.endswith('_9')])


Columns ending in _8:
['last_date_of_month_8', 'arpu_8', 'onnet_mou_8', 'offnet_mou_8', 'roam_ic_mou_8', 'roam_og_mou_8', 'loc_og_t2t_mou_8', 'loc_og_t2m_mou_8', 'loc_og_t2f_mou_8', 'loc_og_t2c_mou_8', 'loc_og_mou_8', 'std_og_t2t_mou_8', 'std_og_t2m_mou_8', 'std_og_t2f_mou_8', 'std_og_t2c_mou_8', 'std_og_mou_8', 'isd_og_mou_8', 'spl_og_mou_8', 'og_others_8', 'total_og_mou_8', 'loc_ic_t2t_mou_8', 'loc_ic_t2m_mou_8', 'loc_ic_t2f_mou_8', 'loc_ic_mou_8', 'std_ic_t2t_mou_8', 'std_ic_t2m_mou_8', 'std_ic_t2f_mou_8', 'std_ic_t2o_mou_8', 'std_ic_mou_8', 'total_ic_mou_8', 'spl_ic_mou_8', 'isd_ic_mou_8', 'ic_others_8', 'total_rech_num_8', 'total_rech_amt_8', 'max_rech_amt_8', 'date_of_last_rech_8', 'last_day_rch_amt_8', 'date_of_last_rech_data_8', 'total_rech_data_8', 'max_rech_data_8', 'count_rech_2g_8', 'count_rech_3g_8', 'av_rech_amt_data_8', 'vol_2g_mb_8', 'vol_3g_mb_8', 'arpu_3g_8', 'arpu_2g_8', 'night_pck_user_8', 'monthly_2g_8', 'sachet_2g_8', 'monthly_3g_8', 'sachet_3g_8', 'fb_user_8']



In [44]:
june_cols = [f"{feature}_6" for feature in common_features]
july_cols = [f"{feature}_7" for feature in common_features]
august_cols = [f"{feature}_8" for feature in common_features]

print("June columns:", len(june_cols))
print("July columns:", len(july_cols))
print("August columns:", len(august_cols))

June columns: 53
July columns: 53
August columns: 53


In [45]:
june_data = df[june_cols].copy()
july_data = df[july_cols].copy()
august_data = df[august_cols].copy()

print("June shape:", june_data.shape)
print("July shape:", july_data.shape)
print("August shape:", august_data.shape)

June shape: (99999, 53)
July shape: (99999, 53)
August shape: (99999, 53)


In [46]:
june_data.columns = common_features
july_data.columns = common_features
august_data.columns = common_features

print(june_data.columns[:10].tolist())

['arpu', 'arpu_2g', 'arpu_3g', 'av_rech_amt_data', 'count_rech_2g', 'count_rech_3g', 'date_of_last_rech', 'date_of_last_rech_data', 'fb_user', 'ic_others']


In [47]:
june_data = june_data.fillna(0)
july_data = july_data.fillna(0)
august_data = august_data.fillna(0)

print("June missing values:", june_data.isnull().sum().sum())
print("July missing values:", july_data.isnull().sum().sum())
print("August missing values:", august_data.isnull().sum().sum())

June missing values: 0
July missing values: 0
August missing values: 0


In [49]:
# ============================================================
# COMMAND 38 — Keep only numeric temporal features
# ============================================================

# Check which common features are non-numeric
non_numeric_features = []

for feature in common_features:
    if not pd.api.types.is_numeric_dtype(df[f"{feature}_6"]):
        non_numeric_features.append(feature)

print("Non-numeric features:")
print(non_numeric_features)

Non-numeric features:
['date_of_last_rech', 'date_of_last_rech_data']


In [54]:
# Create voice activity indicators from raw voice usage

df['voice_active_8'] = (
    (df['total_og_mou_8'].fillna(0) > 0) |
    (df['total_ic_mou_8'].fillna(0) > 0)
).astype(int)

df['voice_active_9'] = (
    (df['total_og_mou_9'].fillna(0) > 0) |
    (df['total_ic_mou_9'].fillna(0) > 0)
).astype(int)

print("voice_active_8 created")
print("voice_active_9 created")

print("\nAugust voice activity:")
print(df['voice_active_8'].value_counts())

print("\nSeptember voice activity:")
print(df['voice_active_9'].value_counts())

voice_active_8 created
voice_active_9 created

August voice activity:
voice_active_8
1    91215
0     8784
Name: count, dtype: int64

September voice activity:
voice_active_9
1    89110
0    10889
Name: count, dtype: int64


In [56]:
# Same target definition used in the original project

df['newly_inactive'] = (
    (df['voice_active_8'] == 1) &
    (df['voice_active_9'] == 0)
).astype(int)

print("newly_inactive created successfully.")

print("\nTarget distribution:")
print(df['newly_inactive'].value_counts())

print("\nTarget percentage:")
print(
    (df['newly_inactive'].value_counts(normalize=True) * 100).round(3)
)

newly_inactive created successfully.

Target distribution:
newly_inactive
0    96004
1     3995
Name: count, dtype: int64

Target percentage:
newly_inactive
0    96.005
1     3.995
Name: proportion, dtype: float64


In [57]:
X_june = june_data.to_numpy(dtype=np.float32)
X_july = july_data.to_numpy(dtype=np.float32)
X_august = august_data.to_numpy(dtype=np.float32)

y_temporal = df["newly_inactive"].to_numpy(dtype=np.int32)

print("June:", X_june.shape)
print("July:", X_july.shape)
print("August:", X_august.shape)
print("Target:", y_temporal.shape)

June: (99999, 51)
July: (99999, 51)
August: (99999, 51)
Target: (99999,)


In [58]:
X_temporal = np.stack(
    [X_june, X_july, X_august],
    axis=1
)

print("Temporal dataset shape:", X_temporal.shape)

Temporal dataset shape: (99999, 3, 51)


In [59]:
print("Temporal dataset shape:", X_temporal.shape)
print("Target shape:", y_temporal.shape)

print("\nOne customer:")
print("Sequence shape:", X_temporal[0].shape)

print("\nTarget of first customer:", y_temporal[0])

Temporal dataset shape: (99999, 3, 51)
Target shape: (99999,)

One customer:
Sequence shape: (3, 51)

Target of first customer: 1


In [60]:
unique, counts = np.unique(y_temporal, return_counts=True)

print("Target distribution:")

for value, count in zip(unique, counts):
    print(
        f"{value}: {count} "
        f"({count / len(y_temporal) * 100:.3f}%)"
    )

Target distribution:
0: 96004 (96.005%)
1: 3995 (3.995%)


In [61]:
from sklearn.model_selection import train_test_split

X_temp_train, X_temp_test, y_temp_train, y_temp_test = train_test_split(
    X_temporal,
    y_temporal,
    test_size=0.20,
    random_state=42,
    stratify=y_temporal
)

print("Training X:", X_temp_train.shape)
print("Testing X :", X_temp_test.shape)

print("Training y:", y_temp_train.shape)
print("Testing y :", y_temp_test.shape)

Training X: (79999, 3, 51)
Testing X : (20000, 3, 51)
Training y: (79999,)
Testing y : (20000,)


In [63]:
X_temp_train, X_temp_val, y_temp_train, y_temp_val = train_test_split(
    X_temp_train,
    y_temp_train,
    test_size=0.20,
    random_state=42,
    stratify=y_temp_train
)

print("Training:", X_temp_train.shape)
print("Validation:", X_temp_val.shape)
print("Test:", X_temp_test.shape)

Training: (63999, 3, 51)
Validation: (16000, 3, 51)
Test: (20000, 3, 51)


In [64]:
print("TRAIN")
print(np.bincount(y_temp_train))
print()

print("VALIDATION")
print(np.bincount(y_temp_val))
print()

print("TEST")
print(np.bincount(y_temp_test))

TRAIN
[61442  2557]

VALIDATION
[15361   639]

TEST
[19201   799]


In [65]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_2d = X_temp_train.reshape(
    -1,
    X_temp_train.shape[-1]
)

scaler.fit(train_2d)

print("Scaler fitted successfully.")
print("Number of features:", scaler.n_features_in_)

Scaler fitted successfully.
Number of features: 51


In [66]:
X_temp_train_scaled = scaler.transform(
    X_temp_train.reshape(-1, X_temp_train.shape[-1])
).reshape(X_temp_train.shape)

X_temp_val_scaled = scaler.transform(
    X_temp_val.reshape(-1, X_temp_val.shape[-1])
).reshape(X_temp_val.shape)

X_temp_test_scaled = scaler.transform(
    X_temp_test.reshape(-1, X_temp_test.shape[-1])
).reshape(X_temp_test.shape)

print("Training:", X_temp_train_scaled.shape)
print("Validation:", X_temp_val_scaled.shape)
print("Test:", X_temp_test_scaled.shape)

Training: (63999, 3, 51)
Validation: (16000, 3, 51)
Test: (20000, 3, 51)


In [67]:
print("Training mean:", X_temp_train_scaled.mean())
print("Training std:", X_temp_train_scaled.std())

print("Minimum:", X_temp_train_scaled.min())
print("Maximum:", X_temp_train_scaled.max())

Training mean: -1.5014372e-09
Training std: 0.98019636
Minimum: -7.0621834
Maximum: 243.79703


In [68]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [69]:
print("Available GPUs:")
print(tf.config.list_physical_devices('GPU'))

Available GPUs:
[]


In [70]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_temp_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_temp_train
)

class_weights = dict(zip(classes, weights))

print("Class weights:")
print(class_weights)

Class weights:
{np.int32(0): np.float64(0.5208082419192084), np.int32(1): np.float64(12.514470082127493)}


In [71]:
lstm_model = Sequential([
    LSTM(
        64,
        input_shape=(
            X_temp_train_scaled.shape[1],
            X_temp_train_scaled.shape[2]
        )
    ),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

lstm_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        29,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,809 (124.25 KB)

 Trainable params: 31,809 (124.25 KB)

 Non-trainable params: 0 (0.00 B)

In [72]:
early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True
)

print("Early stopping configured.")

Early stopping configured.


In [73]:
lstm_history = lstm_model.fit(
    X_temp_train_scaled,
    y_temp_train,
    validation_data=(
        X_temp_val_scaled,
        y_temp_val
    ),
    epochs=30,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.5574 - auc: 0.6813 - loss: 0.6439 - val_accuracy: 0.6456 - val_auc: 0.7402 - val_loss: 0.5949
Epoch 2/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.6373 - auc: 0.7560 - loss: 0.5861 - val_accuracy: 0.7222 - val_auc: 0.7857 - val_loss: 0.5073
Epoch 3/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.6765 - auc: 0.7887 - loss: 0.5558 - val_accuracy: 0.6710 - val_auc: 0.7933 - val_loss: 0.5667
Epoch 4/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.6835 - auc: 0.8056 - loss: 0.5383 - val_accuracy: 0.6791 - val_auc: 0.7990 - val_loss: 0.5472
Epoch 5/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.6964 - auc: 0.8122 - loss: 0.5289 - val_accuracy: 0.7069 - val_auc: 0.8031 - val_loss: 0.5157
Epoch 6/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.6989 - auc: 0.8201 - loss: 0.5197 - val_accuracy: 0.7122 - val_auc: 0.8041 - val_loss: 0.5046
Epoch 7/30
250/250 ━━━━━━━━━━━━━━━

In [74]:
print(
    "Epochs actually completed:",
    len(lstm_history.history["loss"])
)

print(
    "Best validation AUC:",
    max(lstm_history.history["val_auc"])
)

Epochs actually completed: 21
Best validation AUC: 0.8151893019676208


In [75]:
lstm_val_prob = lstm_model.predict(
    X_temp_val_scaled,
    verbose=0
).ravel()

print("Validation predictions:", lstm_val_prob.shape)
print("Minimum probability:", lstm_val_prob.min())
print("Maximum probability:", lstm_val_prob.max())

Validation predictions: (16000,)
Minimum probability: 7.828181e-06
Maximum probability: 0.9887521


In [76]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

lstm_roc_auc = roc_auc_score(
    y_temp_val,
    lstm_val_prob
)

lstm_pr_auc = average_precision_score(
    y_temp_val,
    lstm_val_prob
)

print("LSTM Validation ROC-AUC:", round(lstm_roc_auc, 4))
print("LSTM Validation PR-AUC :", round(lstm_pr_auc, 4))

LSTM Validation ROC-AUC: 0.8151
LSTM Validation PR-AUC : 0.21


In [77]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_temp_val,
    lstm_val_prob
)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)

best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print("Best threshold:", round(best_threshold, 4))
print("Best F1:", round(best_f1, 4))
print("Precision:", round(precision[best_idx], 4))
print("Recall:", round(recall[best_idx], 4))

Best threshold: 0.8216
Best F1: 0.2884
Precision: 0.2662
Recall: 0.3146


In [78]:
from sklearn.metrics import confusion_matrix

lstm_val_pred = (
    lstm_val_prob >= best_threshold
).astype(int)

cm = confusion_matrix(
    y_temp_val,
    lstm_val_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[14807   554]
 [  438   201]]


In [79]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_temp_val,
        lstm_val_pred,
        target_names=[
            "Active",
            "Newly Inactive"
        ],
        digits=4
    )
)

                precision    recall  f1-score   support

        Active     0.9713    0.9639    0.9676     15361
Newly Inactive     0.2662    0.3146    0.2884       639

      accuracy                         0.9380     16000
     macro avg     0.6187    0.6392    0.6280     16000
  weighted avg     0.9431    0.9380    0.9405     16000



In [80]:
gru_model = Sequential([
    GRU(
        64,
        input_shape=(
            X_temp_train_scaled.shape[1],
            X_temp_train_scaled.shape[2]
        )
    ),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

gru_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        22,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,577 (96.00 KB)

 Trainable params: 24,577 (96.00 KB)

 Non-trainable params: 0 (0.00 B)

In [81]:
gru_early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True
)

print("GRU early stopping configured.")

GRU early stopping configured.


In [82]:
gru_history = gru_model.fit(
    X_temp_train_scaled,
    y_temp_train,
    validation_data=(
        X_temp_val_scaled,
        y_temp_val
    ),
    epochs=30,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[gru_early_stopping],
    verbose=1
)

Epoch 1/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.6086 - auc: 0.6889 - loss: 0.6367 - val_accuracy: 0.6306 - val_auc: 0.7642 - val_loss: 0.6011
Epoch 2/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.6551 - auc: 0.7673 - loss: 0.5765 - val_accuracy: 0.6676 - val_auc: 0.7895 - val_loss: 0.5589
Epoch 3/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.6662 - auc: 0.7865 - loss: 0.5574 - val_accuracy: 0.6459 - val_auc: 0.7975 - val_loss: 0.5810
Epoch 4/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.6793 - auc: 0.7985 - loss: 0.5450 - val_accuracy: 0.6676 - val_auc: 0.8016 - val_loss: 0.5523
Epoch 5/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.6802 - auc: 0.8055 - loss: 0.5364 - val_accuracy: 0.7272 - val_auc: 0.8051 - val_loss: 0.4821
Epoch 6/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.6940 - auc: 0.8140 - loss: 0.5266 - val_accuracy: 0.7117 - val_auc: 0.8067 - val_loss: 0.5125
Epoch 7/30
250/250 ━━━━━━━━━━━━━━

In [84]:
print(
    "Epochs actually completed:",
    len(gru_history.history["loss"])
)

print(
    "Best validation AUC:",
    max(gru_history.history["val_auc"])
)

Epochs actually completed: 20
Best validation AUC: 0.8157135844230652


In [85]:
gru_val_prob = gru_model.predict(
    X_temp_val_scaled,
    verbose=0
).ravel()

print("Validation predictions:", gru_val_prob.shape)

print("Minimum probability:", gru_val_prob.min())
print("Maximum probability:", gru_val_prob.max())

Validation predictions: (16000,)
Minimum probability: 2.843837e-05
Maximum probability: 0.97024226


In [86]:
gru_roc_auc = roc_auc_score(
    y_temp_val,
    gru_val_prob
)

gru_pr_auc = average_precision_score(
    y_temp_val,
    gru_val_prob
)

print("GRU Validation ROC-AUC:", round(gru_roc_auc, 4))
print("GRU Validation PR-AUC :", round(gru_pr_auc, 4))

GRU Validation ROC-AUC: 0.8157
GRU Validation PR-AUC : 0.2132


In [87]:
precision_gru, recall_gru, thresholds_gru = precision_recall_curve(
    y_temp_val,
    gru_val_prob
)

f1_scores_gru = (
    2 * precision_gru[:-1] * recall_gru[:-1]
    / (
        precision_gru[:-1]
        + recall_gru[:-1]
        + 1e-10
    )
)

best_idx_gru = np.argmax(f1_scores_gru)

gru_best_threshold = thresholds_gru[best_idx_gru]
gru_best_f1 = f1_scores_gru[best_idx_gru]

print("Best GRU threshold:", round(gru_best_threshold, 4))
print("Best GRU F1:", round(gru_best_f1, 4))
print(
    "Precision:",
    round(precision_gru[best_idx_gru], 4)
)
print(
    "Recall:",
    round(recall_gru[best_idx_gru], 4)
)

Best GRU threshold: 0.8389
Best GRU F1: 0.2818
Precision: 0.264
Recall: 0.302


In [88]:
gru_val_pred = (
    gru_val_prob >= gru_best_threshold
).astype(int)

print("GRU Confusion Matrix:")

print(
    confusion_matrix(
        y_temp_val,
        gru_val_pred
    )
)

GRU Confusion Matrix:
[[14823   538]
 [  446   193]]


In [89]:
print(
    classification_report(
        y_temp_val,
        gru_val_pred,
        target_names=[
            "Active",
            "Newly Inactive"
        ],
        digits=4
    )
)

                precision    recall  f1-score   support

        Active     0.9708    0.9650    0.9679     15361
Newly Inactive     0.2640    0.3020    0.2818       639

      accuracy                         0.9385     16000
     macro avg     0.6174    0.6335    0.6248     16000
  weighted avg     0.9426    0.9385    0.9405     16000



In [90]:
comparison = pd.DataFrame({
    'model': ['XGBoost', 'LSTM', 'GRU'],
    'roc_auc': [
        0.908923,
        lstm_roc_auc,
        gru_roc_auc
    ],
    'pr_auc': [
        0.371770,
        lstm_pr_auc,
        gru_pr_auc
    ],
    'best_f1': [
        0.421227,
        best_f1,
        gru_best_f1
    ],
    'precision': [
        0.377228,
        precision[best_idx],
        precision_gru[best_idx_gru]
    ],
    'recall': [
        0.476846,
        recall[best_idx],
        recall_gru[best_idx_gru]
    ]
})

print(comparison.round(4))

     model  roc_auc  pr_auc  best_f1  precision  recall
0  XGBoost   0.9089  0.3718   0.4212     0.3772  0.4768
1     LSTM   0.8151  0.2100   0.2884     0.2662  0.3146
2      GRU   0.8157  0.2132   0.2818     0.2640  0.3020


In [92]:
import os

os.makedirs(
    '/content/project_artifacts/results',
    exist_ok=True
)

os.makedirs(
    '/content/project_artifacts/models',
    exist_ok=True
)

os.makedirs(
    '/content/project_artifacts/figures',
    exist_ok=True
)

print("Project artifact folders created.")

Project artifact folders created.


In [93]:
comparison_path = '/content/project_artifacts/results/temporal_model_comparison.csv'

comparison.to_csv(
    comparison_path,
    index=False
)

print("Comparison saved:")
print(comparison_path)

Comparison saved:
/content/project_artifacts/results/temporal_model_comparison.csv


In [94]:
lstm_test_prob = lstm_model.predict(
    X_temp_test_scaled,
    verbose=0
).ravel()

lstm_test_roc_auc = roc_auc_score(
    y_temp_test,
    lstm_test_prob
)

lstm_test_pr_auc = average_precision_score(
    y_temp_test,
    lstm_test_prob
)

print("LSTM Test ROC-AUC:", round(lstm_test_roc_auc, 4))
print("LSTM Test PR-AUC :", round(lstm_test_pr_auc, 4))

LSTM Test ROC-AUC: 0.803
LSTM Test PR-AUC : 0.2048


In [95]:
gru_test_prob = gru_model.predict(
    X_temp_test_scaled,
    verbose=0
).ravel()

gru_test_roc_auc = roc_auc_score(
    y_temp_test,
    gru_test_prob
)

gru_test_pr_auc = average_precision_score(
    y_temp_test,
    gru_test_prob
)

print("GRU Test ROC-AUC:", round(gru_test_roc_auc, 4))
print("GRU Test PR-AUC :", round(gru_test_pr_auc, 4))

GRU Test ROC-AUC: 0.8108
GRU Test PR-AUC : 0.2014


In [96]:
lstm_test_pred = (
    lstm_test_prob >= best_threshold
).astype(int)

gru_test_pred = (
    gru_test_prob >= gru_best_threshold
).astype(int)

print("LSTM Test Confusion Matrix:")
print(
    confusion_matrix(
        y_temp_test,
        lstm_test_pred
    )
)

print("\nGRU Test Confusion Matrix:")
print(
    confusion_matrix(
        y_temp_test,
        gru_test_pred
    )
)

LSTM Test Confusion Matrix:
[[18467   734]
 [  555   244]]

GRU Test Confusion Matrix:
[[18493   708]
 [  561   238]]


In [97]:
temporal_metrics = {
    "LSTM": {
        "validation_roc_auc": float(lstm_roc_auc),
        "validation_pr_auc": float(lstm_pr_auc),
        "validation_best_f1": float(best_f1),
        "validation_threshold": float(best_threshold),
        "test_roc_auc": float(lstm_test_roc_auc),
        "test_pr_auc": float(lstm_test_pr_auc)
    },

    "GRU": {
        "validation_roc_auc": float(gru_roc_auc),
        "validation_pr_auc": float(gru_pr_auc),
        "validation_best_f1": float(gru_best_f1),
        "validation_threshold": float(gru_best_threshold),
        "test_roc_auc": float(gru_test_roc_auc),
        "test_pr_auc": float(gru_test_pr_auc)
    }
}

temporal_metrics_path = (
    '/content/project_artifacts/results/'
    'temporal_model_metrics.json'
)

with open(temporal_metrics_path, 'w') as f:
    json.dump(
        temporal_metrics,
        f,
        indent=4
    )

print("Temporal metrics saved:")
print(temporal_metrics_path)

Temporal metrics saved:
/content/project_artifacts/results/temporal_model_metrics.json


In [98]:
os.makedirs(
    '/content/project_artifacts/models',
    exist_ok=True
)

lstm_model.save(
    '/content/project_artifacts/models/lstm_temporal_model.keras'
)

gru_model.save(
    '/content/project_artifacts/models/gru_temporal_model.keras'
)

print("LSTM model saved.")
print("GRU model saved.")

LSTM model saved.
GRU model saved.


In [100]:
import joblib

In [101]:
joblib.dump(
    scaler,
    '/content/project_artifacts/models/temporal_scaler.pkl'
)

print("Temporal scaler saved.")

Temporal scaler saved.


In [102]:
import os

for root, dirs, files in os.walk('/content/project_artifacts'):
    for file in files:
        print(os.path.join(root, file))

/content/project_artifacts/results/temporal_model_metrics.json
/content/project_artifacts/results/temporal_model_comparison.csv
/content/project_artifacts/models/temporal_scaler.pkl
/content/project_artifacts/models/lstm_temporal_model.keras
/content/project_artifacts/models/gru_temporal_model.keras


In [103]:
import os

print("===== MODELS =====")
for f in os.listdir('/content/project_artifacts/models'):
    print(f)

print("\n===== RESULTS =====")
for f in os.listdir('/content/project_artifacts/results'):
    print(f)

print("\n===== FIGURES =====")
for f in os.listdir('/content/project_artifacts/figures'):
    print(f)

===== MODELS =====
temporal_scaler.pkl
lstm_temporal_model.keras
gru_temporal_model.keras

===== RESULTS =====
temporal_model_metrics.json
temporal_model_comparison.csv

===== FIGURES =====


In [104]:
import os

print(os.path.exists('/content/telecom-churnprediction'))

if os.path.exists('/content/telecom-churnprediction'):
    print(os.listdir('/content/telecom-churnprediction'))

True
['results', 'README.md', '03_temporal_models.ipynb', 'figures', 'notebooks', 'requirements.txt', '.git', '.gitignore', 'telecom_churn_data.csv']


In [105]:
import shutil
import os

repo = '/content/telecom-churn-prediction'

os.makedirs(f'{repo}/models', exist_ok=True)
os.makedirs(f'{repo}/results', exist_ok=True)

# Copy models
shutil.copy(
    '/content/project_artifacts/models/lstm_temporal_model.keras',
    f'{repo}/models/lstm_temporal_model.keras'
)

shutil.copy(
    '/content/project_artifacts/models/gru_temporal_model.keras',
    f'{repo}/models/gru_temporal_model.keras'
)

shutil.copy(
    '/content/project_artifacts/models/temporal_scaler.pkl',
    f'{repo}/models/temporal_scaler.pkl'
)

# Copy results
shutil.copy(
    '/content/project_artifacts/results/temporal_model_comparison.csv',
    f'{repo}/results/temporal_model_comparison.csv'
)

shutil.copy(
    '/content/project_artifacts/results/temporal_model_metrics.json',
    f'{repo}/results/temporal_model_metrics.json'
)

print("Temporal model files copied successfully.")

Temporal model files copied successfully.


In [106]:
for root, dirs, files in os.walk('/content/telecom-churnprediction'):
    for file in files:
        if (
            'lstm' in file.lower()
            or 'gru' in file.lower()
            or 'temporal' in file.lower()
        ):
            print(os.path.join(root, file))

/content/telecom-churnprediction/03_temporal_models.ipynb
/content/telecom-churnprediction/notebooks/03_temporal_models.ipynb


In [107]:
%cd /content/telecom-churnprediction
!git status

/content/telecom-churnprediction
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	telecom_churn_data.csv

nothing added to commit but untracked files present (use "git add" to track)


In [108]:
import os

repo = "/content/telecom-churnprediction"

print("Repository contents:")
for item in os.listdir(repo):
    print(item)

Repository contents:
results
README.md
03_temporal_models.ipynb
figures
notebooks
requirements.txt
.git
.gitignore
telecom_churn_data.csv


In [109]:
for folder in ["models", "results", "notebooks"]:
    path = f"/content/telecom-churnprediction/{folder}"

    print(f"\n===== {folder} =====")

    if os.path.exists(path):
        print(os.listdir(path))
    else:
        print("Folder does not exist")


===== models =====
Folder does not exist

===== results =====
['shap_feature_importance.csv', 'customer_risk_predictions.csv', 'model_features.csv', 'model_metrics.json', 'risk_group_summary.csv']

===== notebooks =====
['01_data_preprocessing.ipynb', '03_temporal_models.ipynb']


In [110]:
import shutil
import os

repo = "/content/telecom-churnprediction"

os.makedirs(f"{repo}/models", exist_ok=True)
os.makedirs(f"{repo}/results", exist_ok=True)

files_to_copy = [
    (
        "/content/project_artifacts/models/lstm_temporal_model.keras",
        f"{repo}/models/lstm_temporal_model.keras"
    ),
    (
        "/content/project_artifacts/models/gru_temporal_model.keras",
        f"{repo}/models/gru_temporal_model.keras"
    ),
    (
        "/content/project_artifacts/models/temporal_scaler.pkl",
        f"{repo}/models/temporal_scaler.pkl"
    ),
    (
        "/content/project_artifacts/results/temporal_model_comparison.csv",
        f"{repo}/results/temporal_model_comparison.csv"
    ),
    (
        "/content/project_artifacts/results/temporal_model_metrics.json",
        f"{repo}/results/temporal_model_metrics.json"
    )
]

for source, destination in files_to_copy:
    if os.path.exists(source):
        shutil.copy2(source, destination)
        print("Copied:", os.path.basename(source))
    else:
        print("MISSING:", source)

Copied: lstm_temporal_model.keras
Copied: gru_temporal_model.keras
Copied: temporal_scaler.pkl
Copied: temporal_model_comparison.csv
Copied: temporal_model_metrics.json


In [111]:
%cd /content/telecom-churnprediction
!git status

/content/telecom-churnprediction
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	models/
	results/temporal_model_comparison.csv
	results/temporal_model_metrics.json
	telecom_churn_data.csv

nothing added to commit but untracked files present (use "git add" to track)


In [112]:
!git add models/
!git add results/temporal_model_comparison.csv
!git add results/temporal_model_metrics.json

In [113]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   models/gru_temporal_model.keras
	new file:   models/lstm_temporal_model.keras
	new file:   results/temporal_model_comparison.csv
	new file:   results/temporal_model_metrics.json

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	telecom_churn_data.csv



In [114]:
!git commit -m "Add temporal LSTM and GRU churn models"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@52db5d075ea7.(none)')


In [115]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [117]:
!sudo apt-get update -qq
!sudo apt-get install gh -y -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [121]:
!git remote -v

origin	https://github.com/Henix285/telecom-churnprediction.git (fetch)
origin	https://github.com/Henix285/telecom-churnprediction.git (push)


In [122]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [123]:
import getpass

token = getpass.getpass("Paste your GitHub token: ")

Paste your GitHub token: ··········


In [124]:
import subprocess

remote = "https://Henix285:" + token + "@github.com/Henix285/telecom-churnprediction.git"

subprocess.run(
    ["git", "push", remote, "main"],
    check=True
)

print("Successfully pushed to GitHub.")

Successfully pushed to GitHub.


In [125]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   models/gru_temporal_model.keras
	new file:   models/lstm_temporal_model.keras
	new file:   results/temporal_model_comparison.csv
	new file:   results/temporal_model_metrics.json

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	telecom_churn_data.csv

